# LG-CXR FRCNN Baseline

Colab-first workflow for the AMIA Public Challenge 2026 Faster R-CNN baseline. V1 trains only the torchvision Faster R-CNN scanner and creates a valid Kaggle `submission.csv`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Open Project Folder

This supports either a project folder stored in Drive or a folder uploaded manually to `/content`.

In [ ]:
import os
from pathlib import Path

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/amia-lgcxr-frcnn')
LOCAL_PROJECT_DIR = Path('/content/amia-lgcxr-frcnn')

PROJECT_DIR = DRIVE_PROJECT_DIR if DRIVE_PROJECT_DIR.exists() else LOCAL_PROJECT_DIR
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Project folder not found at {DRIVE_PROJECT_DIR} or {LOCAL_PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('Current directory =', Path.cwd())

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Check Environment

In [ ]:
import platform
import torch
import torchvision

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    x = torch.ones((2, 2), device='cuda')
    print('CUDA tensor test:', x.sum().item())

## 5. Configure Paths

The scripts read `configs/baseline_frcnn.yaml` and also honor these environment variables. Edit `DATA_ROOT` if your dataset is somewhere else.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = str(Path.cwd())
DATA_ROOT = '/content/drive/MyDrive/datasets/amia-public-challenge-2026'
WORK_DIR = '/content/drive/MyDrive/amia-lgcxr-frcnn/outputs'

# Kaggle-style alternatives:
# DATA_ROOT = '/kaggle/input/amia-public-challenge-2026'
# WORK_DIR = '/kaggle/working'

os.environ['LGCXR_DATA_ROOT'] = DATA_ROOT
os.environ['LGCXR_WORK_DIR'] = WORK_DIR
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_ROOT =', DATA_ROOT)
print('WORK_DIR =', WORK_DIR)

## 6. Run Preflight

In [ ]:
!python scripts/00_preflight.py --config configs/baseline_frcnn.yaml

## 7. Audit Dimensions

Run this before changing `scanner.image_size` or `scanner.max_size`. The audit also checks the original-coordinate issue: `train.csv` boxes are in original scan space, while the PNG files may be resized.

In [ ]:
!python scripts/05_audit_dimensions.py --config configs/baseline_frcnn.yaml

## 8. Smoke Test

This trains for at most one epoch on small subsets and creates a format-valid submission.

In [ ]:
!python scripts/04_full_pipeline.py --config configs/baseline_frcnn.yaml --smoke-test

## 9. Full Training

In [ ]:
!python scripts/01_train_scanner.py --config configs/baseline_frcnn.yaml

## 10. Inference

In [ ]:
!python scripts/02_predict_scanner.py --config configs/baseline_frcnn.yaml

## 11. Make Submission

In [ ]:
!python scripts/03_make_submission.py --config configs/baseline_frcnn.yaml

## 12. Full Pipeline

In [ ]:
!python scripts/04_full_pipeline.py --config configs/baseline_frcnn.yaml

## 13. Inspect Outputs

In [ ]:
from pathlib import Path

for path in sorted(Path(os.environ['LGCXR_WORK_DIR']).glob('*')):
    print(path)